# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishigupgta1234-ux/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

### Research Question

Which pages should a content team review first when they have a limited weekly review capacity, and can a simple learned model prioritize declining pages better than a hand-written triage rule?

### Decision

The goal is to rank pages so that a content team can decide which pages to inspect first for a possible refresh.

### Unit of Analysis

The unit of analysis is one page-level record.

### Output

The final output is a ranked list of pages with a model score and action/reason information that can support human review.

### Human Action

An editor or content team member can review the highest-ranked pages first and decide whether a refresh or other content action is appropriate.

### Why Data/ML Helps

A hand-written rule provides a simple baseline, while the decision-tree model can combine multiple page-level signals to produce a more consistent ranking.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

### Dataset Used

The model training and evaluation use the anonymized FlyRank ML Internship starter dataset. It contains roughly 30,000 content-page records with 44 columns and a 90-day rolling performance window per page.

Earlier data-contract work also used the larger internship warehouse to check the data grain, availability, and feature timing. The modeling work itself uses the smaller anonymized starter release.

### Data Safety and Exclusions

The analysis does not include client names, raw URLs, private search queries, credentials, or other client-identifying information.

Rows without confirmed search-data availability were excluded instead of being treated as zero performance, because missing search data does not mean that a page had zero visibility.

Pseudonymous client and page identifiers are used only for grouping and data handling. They are not used as model features.

### Leakage Prevention

The target is a proxy for a declining page, based on `trend_direction == "down"`.

The columns `trend_direction` and `trend_pct` were excluded from the model features because they are directly related to the target and could cause target leakage.

The final evaluation uses a client-level train/test split so that pages from the same client do not appear in both training and test data.

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### Target / Proxy Label

There is no direct ground-truth label for whether a page truly needs a content refresh. Therefore, I use a proxy label called `declining`.

A page is labeled as declining when `trend_direction == "down"`.

### Features

The model uses 17 page-level signals available before the outcome, covering:

- impressions
- clicks
- sessions
- position
- CTR
- scroll rate
- engagement rate
- word count
- character count
- content age
- days since last update
- and related page-level performance and content signals

The exact feature set was fixed before evaluation.

### Deliberate Exclusions

`trend_direction` and `trend_pct` were not used as model features because they are directly related to the target and could create target leakage.

Client and page identifiers were also not used as predictive features.

### Baseline

The Week-4 baseline is a hand-written scoring rule based mainly on content staleness and visibility. A page receives higher priority when it is at least 180 days old since its last update and has at least 500 impressions in the available performance window.

### Model

I trained a shallow Decision Tree with `max_depth=5`.

The model's predicted probability of the declining class is used as the ranking score.

### Validation

The data was split at the client level rather than randomly at the row level. This keeps all pages belonging to a client in the same split and prevents the same client's pages from appearing in both training and testing.

The final split contained 26,581 training rows from 25 clients and 3,419 test rows from 7 clients, with zero client overlap.

The random seed used was 42.

### Evaluation Metric

The main evaluation metric is Precision@50 because the practical use case is to give a content team a small ranked list of pages to review first.

The model is compared with the Week-4 baseline on the same client-level test set.

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

### Evaluation Results

Both methods were evaluated on the same client-level held-out test set using Precision@50.

Precision@50 measures the fraction of true declining pages among the top 50 pages ranked by each method. This metric matches the practical use case because the content team has limited review capacity and needs a small, prioritized queue.

| Method | Precision@50 | Validation |
|---|---:|---|
| Hand-written baseline | 0.38 | Client-level holdout |
| Decision Tree (depth 5) | 0.68 | Client-level holdout |

The Decision Tree therefore showed higher observed Precision@50 than the hand-written baseline on this test split.

### Base-Rate Context

The majority-class base rate on the test set was **52.38%**. This provides additional context for interpreting the ranking result and avoids presenting Precision@50 without the underlying class distribution.

### Error Analysis

The model's errors were concentrated around pages with less extreme combinations of age, visibility, and performance signals. These borderline cases are difficult to separate using the proxy label because a declining trend does not necessarily mean that a page truly requires a refresh.

For this reason, the ranked output should be treated as a review-prioritization tool rather than an automatic decision system.

### Interpretation

The result is directional and sample-specific: on this client-level holdout, the learned model ranked declining pages more effectively at the top of the queue than the hand-written baseline.

## 5. Limitations

*What this work cannot claim.*

### Limitations

The target is a proxy rather than a direct measure of whether a page actually needed or benefited from a content refresh. A page showing a declining trend does not automatically mean that refreshing its content will improve performance.

The evaluation is based on one client-level holdout from the available internship dataset. Therefore, the observed Precision@50 result should not be treated as a guarantee of performance on new clients, different datasets, or future time periods.

The model ranks pages for review; it does not determine which pages should definitely be refreshed.

### Honest Framing

The Decision Tree showed an observed improvement over the hand-written baseline on this particular client-level holdout: Precision@50 was 0.68 compared with 0.38 for the baseline.

This is a measured, directional result and should be interpreted as decision-support evidence rather than causal evidence that the model will improve SEO or content performance.

Human review remains necessary before taking any content action.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

### Ranked Recommendations

The model output is used to create a prioritized review queue for a content team with limited weekly review capacity.

1. **Review the highest-ranked pages first**
   - Start with pages receiving the highest model scores.
   - Use the ranking to focus human review on the strongest observed candidates.

2. **Prioritize stale, high-visibility pages**
   - Pages that are older and have meaningful visibility should receive early attention.
   - These pages can represent useful opportunities for content review.

3. **Review low-CTR pages with strong positions**
   - Pages with relatively strong search positions but weaker CTR can be reviewed for possible title, snippet, or content-presentation improvements.

4. **Use the reason/action information**
   - The reason code from the action playbook helps the reviewer understand why a page was prioritized.
   - The reviewer should verify the page context before taking action.

### Recommended Workflow

The content team can review the ranked list from the top, inspect the model score and reason code, and decide whether a refresh or another content action is appropriate.

The model is used for **decision support only**. It does not automatically rewrite, publish, delete, or modify content.

The ranking should be revalidated on additional clients or future data before being treated as a general production rule.

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

### Artifacts Included in the Paper

The deployed paper includes a Precision@50 comparison chart and a results table comparing the hand-written baseline with the Decision Tree model on the same client-level held-out test set.

The paper also reports the 52.38% majority-class base rate to provide context for the Precision@50 results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
